# 🌍 Microsoft Planetary Computer Pro - Python SDK Tutorial
Welcome to the official tutorial for using the **Microsoft Planetary Computer Pro Python SDK**.

This guide walks you through:
- 🔐 Authentication and client setup
- 🧱 STAC collection creation
- 🖼️ Uploading and accessing thumbnails
- 🔍 Querying and reading STAC collections
- 📦 Ingesting STAC items from public catalogs
- 🧭 Render and mosaic configuration
- 🧪 Querying ingestion status
- 🗑️ Deleting items and collections

All steps use the Python SDK only (no REST API).

In [2]:
!az login

[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "id": "8cff5c8a-98f3-44ad-b300-2d44716c802c",
    "isDefault": false,
    "managedByTenants": [
      {
        "tenantId": "2f4a9838-26b7-47ee-be60-ccc1fdec5953"
      }
    ],
    "name": "Service 360 Test",
    "state": "Enabled",
    "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "user": {
      "name": "haseidfa@microsoft.com",
      "type": "user"
    }
  },
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "id": "a8dc551f-cbe8-47e9-87c1-d9570ac6d69d",
    "isDefault": false,
    "managedByTenants": [],
    "name": "OFP-TPA-markti",
    "state": "Enabled",
    "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "user": {
      "name": "haseidfa@microsoft.com",
      "type": "user"
    }
  },
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "id": "54b875cc-a81a-4914-8bfd-

## 🔐 Authenticate and Create Client

In [14]:
from azure.identity import DefaultAzureCredential
from azure.planetarycomputer import MicrosoftPlanetaryComputerProClient
from datetime import datetime
import uuid

credential = DefaultAzureCredential()

token = credential.get_token("https://geocatalog.spatio.azure.com/.default").token

endpoint = "https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net"
client = MicrosoftPlanetaryComputerProClient(endpoint=endpoint, credential=DefaultAzureCredential())
collection_id = f"tutorial-collection-{datetime.now().strftime('%Y%m%d%H%M%S')}"
print("✅ Client initialized and collection ID generated:", collection_id)


✅ Client initialized and collection ID generated: tutorial-collection-20250515125901


## 🧱 Create a STAC Collection

In [15]:
collection_payload = {
    "description": "Tutorial collection for integration tests",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}
    },
    "id": collection_id,
    "license": "CC-BY-4.0",
    "links": [],
    "stac_version": "1.0.0",
    "title": "Tutorial Collection",
    "type": "Collection",
    "item_assets": {
        "GEC": {
            "type": "image/tiff; application=geotiff; profile=cloud-optimized",
            "roles": ["data"],
            "title": "VV polarization",
            "description": "Gamma naught values corrected for terrain",
            "raster:bands": [{
                "nodata": -32768,
                "data_type": "uint8",
                "spatial_resolution": 0.477
            }]
        }
    }
}

response = client.stac_collection_operations.begin_create(body=collection_payload, polling=False)
print(f"✅ Collection created: {collection_id}")


✅ Collection created: tutorial-collection-20250515125901


## 📖 Read the Collection

In [22]:
collection = client.stac_collection_operations.get(collection_id=collection_id)
print("✅ Collection read successfully")
print(collection.as_dict())


✅ Collection read successfully
{'id': 'tutorial-collection-20250515125901', 'type': 'Collection', 'links': [{'rel': 'items', 'type': 'application/geo+json', 'href': 'https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net/stac/collections/tutorial-collection-20250515125901/items'}, {'rel': 'parent', 'type': 'application/json', 'href': 'https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net/stac/'}, {'rel': 'root', 'type': 'application/json', 'href': 'https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net/stac/'}, {'rel': 'self', 'type': 'application/json', 'href': 'https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net/stac/collections/tutorial-collection-20250515125901'}], 'title': 'Tutorial Collection', 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]}, 'temporal': {'interval': [['2020-01-01T00:00:00Z', None]]}}, 'license': 'CC-BY-4.0', 'description': 'Tutor

In [29]:
# 🖼️ Save thumbnail URL before removing assets
import requests

pc_collection = "sentinel-2-l2a"
response = requests.get(f"https://planetarycomputer.microsoft.com/api/stac/v1/collections/{pc_collection}")
response.raise_for_status()
stac_collection = response.json()

# ✅ Store the thumbnail URL for later use
thumbnail_url = stac_collection['assets']['thumbnail']['href']
print("✅ Thumbnail URL to reuse later:", thumbnail_url)

# ❌ Remove assets before uploading to GeoCatalog
stac_collection.pop('assets')
print("🗑️ 'assets' field removed from STAC Collection JSON.")


✅ Thumbnail URL to reuse later: https://ai4edatasetspublicassets.blob.core.windows.net/assets/pc_thumbnails/sentinel-2.png
🗑️ 'assets' field removed from STAC Collection JSON.


In [39]:
import json
from io import BytesIO
from PIL import Image
import requests
from azure.planetarycomputer import MicrosoftPlanetaryComputerProClient
from azure.identity import DefaultAzureCredential

# Setup
credential = DefaultAzureCredential()
endpoint = "https://ppe-ch-2.ceefe5e6ahe2haft.northcentralus.geocatalog.spatio-ppe.azure-test.net"
client = MicrosoftPlanetaryComputerProClient(endpoint=endpoint, credential=credential)

collection_id = "tutorial-collection-20250514165000"
thumbnail_url = "https://ai4edatasetspublicassets.blob.core.windows.net/assets/pc_thumbnails/sentinel-2.png"

# Define thumbnail asset metadata
data_str = json.dumps({
    "key": "thumbnail",
    "href": thumbnail_url,
    "type": "image/png",
    "roles": ["thumbnail"],
    "title": "Sentinel-2 preview"
})

# Download thumbnail
thumbnail_bytes = BytesIO(requests.get(thumbnail_url).content)
thumbnail_tuple = ("thumbnail.png", thumbnail_bytes)

# Upload using correct client group
try:
    client.stac_collection_assets.collection_asset(collection_id, data=data_str, file=thumbnail_tuple)
    print("✅ Thumbnail uploaded successfully.")
except Exception as e:
    print("❌ Failed to upload thumbnail:", e)


❌ Failed to upload thumbnail: 'StacCollectionAssetsOperations' object has no attribute 'collection_asset'


## 🖼️ Fetch and Display Thumbnail

In [10]:
from PIL import Image
from io import BytesIO

try:
    thumbnail_stream = client.stac_collection_thumbnails.get(collection_id=collection_id)
    thumbnail_bytes = b''.join([chunk for chunk in thumbnail_stream])
    img = Image.open(BytesIO(thumbnail_bytes))
    img.show()
    print("✅ Thumbnail fetched and rendered.")
except Exception as e:
    print("❌ Thumbnail fetch failed:", e)


❌ Thumbnail fetch failed: (ResourceNotFound) Asset key 'thumbnail' not found in collection 'tutorial-collection-20250515112010'. Please verify the asset key and collection ID, and try again.
Code: ResourceNotFound
Message: Asset key 'thumbnail' not found in collection 'tutorial-collection-20250515112010'. Please verify the asset key and collection ID, and try again.


## 🔍 Query the Planetary Computer

In [ ]:
search_body = {
    "collections": [collection_id],
    "bbox": [-70.9, -33.5, -70.7, -33.3],
    "datetime": "2023-01-01T00:00:00Z/2023-12-31T23:59:59Z"
}

response = client.stac_search_operations.create(body=search_body)
print("✅ STAC search results:")
print(response.as_dict())


## 📦 Ingest STAC Items

In [ ]:
from azure.planetarycomputer.models import StacItem
import pystac
import uuid
import requests

# Replace with your signed STAC catalog URL
catalog_url = "<PUT_SIGNED_CATALOG_URL_HERE>"
response = requests.get(catalog_url)
catalog_dict = response.json()
catalog = pystac.Catalog.from_dict(catalog_dict)

items_to_post = []
for item in catalog.get_all_items():
    item_dict = item.to_dict()
    item_dict["collection"] = collection_id
    item_dict["id"] = str(uuid.uuid4())
    items_to_post.append(item_dict)

print(f"✅ Prepared {len(items_to_post)} items for ingestion.")

# 🚀 Post items via SDK
operation = client.stac_items.create_or_replace(
    collection_id=collection_id,
    body={"features": items_to_post}
)
print("✅ Ingestion started (asynchronous operation).")


## 🧭 Configure Render Options

In [ ]:
render_option = {
    "id": "vv-polarization",
    "name": "VV polarization",
    "description": "VV asset scaled to 0–255 grayscale",
    "type": "raster-tile",
    "options": "assets=GEC&rescale=0,255&colormap_name=gray",
    "minZoom": 8,
    "conditions": [{"property": "sar:polarizations", "value": ["VV"]}]
}

client.stac_collection_render_options.create_or_replace(
    collection_id=collection_id,
    render_option_id=render_option["id"],
    body=render_option
)
print("✅ Render config applied.")


## 🧩 Configure Mosaic Definition

In [ ]:
mosaics = {
    "mosaics": [
        {"id": "default", "name": "Default Mosaic", "description": "", "cql": []}
    ]
}

client.stac_collection_mosaics.create_or_replace(
    collection_id=collection_id,
    mosaic_definition=mosaics
)
print("✅ Mosaic definition created.")


## 📊 Check Ingestion Status

In [ ]:
import json

try:
    operations = list(client.ingestion_operations.list_all(collection_id=collection_id))
    print(f"✅ Found {len(operations)} ingestion operations.")
    for op in operations:
        print(json.dumps(op, indent=2))
except Exception as e:
    print("❌ Failed to fetch ingestion operations:", e)


## 🗑️ Delete a STAC Item

In [ ]:
# Replace with actual item ID from your ingestion
item_id = "<PUT_ITEM_ID_HERE>"

try:
    raw_response = next(client.stac_items._delete_initial(
        collection_id=collection_id,
        item_id=item_id
    ))
    response_text = raw_response.decode("utf-8").strip()
    if not response_text:
        print(f"✅ STAC item '{item_id}' deleted successfully.")
    else:
        print(f"⚠️ Unexpected response:
{response_text}")
except Exception as e:
    print("❌ Failed to delete STAC item:", e)


## 🧹 Delete the STAC Collection

In [ ]:
try:
    poller = client.stac_collection_operations.begin_delete(collection_id=collection_id)
    poller.result()
    print(f"✅ Collection '{collection_id}' deleted.")
except Exception as e:
    print(f"❌ Failed to delete collection '{collection_id}':", e)
